In [114]:
import torch
from modeling import *
import matplotlib.pyplot as plt

In [115]:
def tokenize(text):
    inp_ids = wrapper.tokenize(text)
    str_toks = wrapper.list_decode(inp_ids[0])
    return inp_ids, str_toks

In [117]:
model, tokenizer = load_gpt2('gpt2-medium')
model = model.float()

KeyboardInterrupt: 

In [5]:
wrapper = GPT2Wrapper(model, tokenizer)

In [6]:
poland_text="""Q: What is the capital of France?
A: Paris
Q: What is the capital of Poland?
A:"""

In [21]:
poland_ids, pol_toks = tokenize(poland_text)
logits = wrapper.get_layers(poland_ids)
wrapper.print_top(logits[1:]) #skip the embedding layer

0  ( [ The:,
 at and Act A
1  A The ( [ Is59
 At and40
2  A [ ( The At Is Act at59,
3  A [ ( Act At Is The CH An at
4  A [ At Q (Q The Are M An
5  A M No At The payable Q Qu (Q
6  No M A The C Die An H En Qu
7  C A No The M n P N H An
8  A The C P H No n Ass N T
9  A C No nil The Ch P An H N
10  A The G C N P No Me An Le
11  A C N None P G The Pr Ce H
12  Unknown None C G A N Bar The Ch P
13  C P N G B A Unknown St None The
14  St N G P Poland B C Pol A D
15  Poland P St Pol Warsaw Polish N B G Germany
16  Poland Warsaw Polish Poles Budapest Prague Pol Germany Berlin Moscow
17  Poland Warsaw Polish Poles Budapest Prague � Pol Lithuania Moscow
18  Poland Warsaw Polish Prague Budapest Poles Moscow � Berlin Kiev
19  Warsaw Poland Polish Budapest Prague Moscow Berlin Kiev � Frankfurt
20  Warsaw Poland Prague Budapest Polish Moscow Kiev Berlin Frankfurt Brussels
21  Warsaw Poland Polish Prague Budapest � Kiev Sz Berlin Moscow
22  Warsaw Poland Prague Budapest K W Kiev Sz Moscow Berlin
23  W

# Messy Playground

In [8]:
# m
outputs = model(input_ids=poland_ids, output_hidden_states=True)

In [54]:
outputs.hidden_states[0][:, -1, :].shape

torch.Size([1, 1024])

In [48]:
model.lm_head.weight.shape

torch.Size([50257, 1024])

In [55]:
model.transformer.ln_f(outputs.hidden_states[0][:, -1, :]).T.shape

torch.Size([1024, 1])

In [68]:
torch.matmul(model.lm_head.weight, model.transformer.ln_f(outputs.hidden_states[23][:, -1, :]).T)

tensor([[-63.4535],
        [-63.6195],
        [-66.8317],
        ...,
        [-76.2258],
        [-74.0107],
        [-61.3317]], grad_fn=<MmBackward0>)

In [71]:
layers = wrapper.get_layers(poland_ids)

In [72]:
layers.shape

torch.Size([25, 50257])

In [88]:
translation_text="""Q: Translate this phrase from English to German: I ate the apple
A: Ich habe den Apfel gegessen
Q: Translate this phrase from English to German: I made the bed
A: Ich habe das Bett gemacht"""

In [89]:
trans_ids, trans_toks = tokenize(translation_text)

In [90]:
trans_ids_no_answer = trans_ids[:, :-8]

In [91]:
logits = wrapper.get_layers(trans_ids_no_answer)

In [92]:
wrapper.print_top(logits[1:])

0  ( [ The at
 and, A At:
1  ( The [ A
 and at en'Act
2  ( [ A en and The n
, at
3  ( [ en n A Act Letter Is Em,
4  [ ( n Letter en The A q gu before
5  ( n [ Letter A en No q M i
6  ( C n A M No The L i An
7  n ( C No i The N en l,
8  ( n H C No A i The and,
9  n i H A ( No C The en no
10  n i A No Ch The N H Is Qu
11  No i A The Ch Is n id C Di
12  No Is n Ch  A J My N i
13  J  i C My A Is No The Oh
14  Is My Je Il Oh H  N J Ob
15  Die � Il Me Oh Der Je  H Is
16  Die von � Der Das Il Me Ich il Otto
17  Die von Das Der Deutsche German Ich Otto Bundes Ü
18  Die Das von Ich Sie Der Ü Bundes Deutsche �
19  Die Das Ich Sie Der von Bundes Deutsche Aus Er
20  Ich Das Die Der Sie von Aus Er Deutsche Zur
21  Ich Die Das Der Er Sie Aus Eh I 
22  Ich Die Das Der Er I D W H 
23  Ich I Die D Das Der Er W E H


In [98]:
logits_per_token = []
for token_spot in range(-8, -1):
    trans_ids_segment = trans_ids[:, :token_spot]
    logits = wrapper.get_layers(trans_ids_segment)
    logits_per_token.append(logits)

In [99]:
wrapper.print_top(logits_per_token[0][1:])

0  ( [ The at
 and, A At:
1  ( The [ A
 and at en'Act
2  ( [ A en and The n
, at
3  ( [ en n A Act Letter Is Em,
4  [ ( n Letter en The A q gu before
5  ( n [ Letter A en No q M i
6  ( C n A M No The L i An
7  n ( C No i The N en l,
8  ( n H C No A i The and,
9  n i H A ( No C The en no
10  n i A No Ch The N H Is Qu
11  No i A The Ch Is n id C Di
12  No Is n Ch  A J My N i
13  J  i C My A Is No The Oh
14  Is My Je Il Oh H  N J Ob
15  Die � Il Me Oh Der Je  H Is
16  Die von � Der Das Il Me Ich il Otto
17  Die von Das Der Deutsche German Ich Otto Bundes Ü
18  Die Das von Ich Sie Der Ü Bundes Deutsche �
19  Die Das Ich Sie Der von Bundes Deutsche Aus Er
20  Ich Das Die Der Sie von Aus Er Deutsche Zur
21  Ich Die Das Der Er Sie Aus Eh I 
22  Ich Die Das Der Er I D W H 
23  Ich I Die D Das Der Er W E H


In [112]:
wrapper.print_top(logits_per_token[5][1:])

0 partcentpectsisterearchkeyncmitertota
1 partpectsciync en wellcentidesertades
2 centquecipartync en' Elmitch
3 quecentyncci en'miteng well related
4 ynccent'quemit wellci en related e
5 cimitput'ister bascent ass gud
6 cicent assmit Ur eff' bas soync
7 centquemit' Fci ass bas E '
8 mitcent ch g'yncque effcisel
9  damit g kcicent � nque hor
10  d � da g n decent m k far
11  d da g de ncent � m bas y
12  g d m bas da � n hor de car
13  hor d da de m g � n la ch
14  dsel de da m g � hor sch bas
15  dsel sch m mit k � hor de Berlin
16  mit Bundes Berlin sch German d k Sch Bild Ges
17  Bundes Berlin German Sch mit Ges sch Bild Schwe Deutsche
18  Ges Bundes Berlin Sch mit Bild sch Schw Schwe Bed
19  Bundes Ges Sch Spiel Bed Schwe Berlin Schw Buch Bild
20  Bed Buch bed Ges Schwe Sch Bundes Land Spiel sch
21  Bed Buch bed Ap Ge Schwe Land Ges Be Spiel
22  Bed bed Buch Ap Ge Be B Land Sch k
23  Bed bed B Ap Be K Buch Ge k Sch
